# 扫雷法学硕士 - 定制 GRPO 培训

## 目标
使用 GRPO 通过 LoRA 微调 LLM 来玩扫雷，方法是：
- **输入**：JSON 游戏状态（棋盘配置）
- **输出**：JSON 操作（显示或标记单元格）

各支队伍将竞相培养最优秀的扫雷法学硕士！

## 训练方法
- **型号**：带 LoRA 的 GPT-OSS 20B
- **方法**：GRPO（组相关策略优化）
- **框架**：Unsloth（速度提高 2-6 倍，VRAM 减少 70%）
- **硬件**：AMD GPU (ROCm)

# 安装

安装针对 AMD GPU 优化的 Unsloth 和依赖项：

In [ ]:
%%bash
python -m pip install -qU uv --root-user-action=ignore

ROCM_TAG="$({ command -v amd-smi >/dev/null 2>&1 && amd-smi version 2>/dev/null | awk -F'ROCm version: ' 'NF>1{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { [ -r /opt/rocm/.info/version ] && awk -F. '{print "rocm"$1"."$2; exit}' /opt/rocm/.info/version; } || { command -v hipconfig >/dev/null 2>&1 && hipconfig --version 2>/dev/null | awk -F': *' '/HIP version/{split($2,a,"."); print "rocm"a[1]"."a[2]; ok=1; exit} END{exit !ok}'; } || { command -v dpkg-query >/dev/null 2>&1 && ver="$(dpkg-query -W -f='${Version}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; } || { command -v rpm >/dev/null 2>&1 && ver="$(rpm -q --qf '%{VERSION}\n' rocm-core 2>/dev/null)" && [ -n "$ver" ] && awk -F'[.-]' '{print "rocm"$1"."$2; exit}' <<<"$ver"; })"
[ -n "$ROCM_TAG" ] || { echo "Could not detect ROCm. Install ROCm first or set ROCM_TAG manually."; exit 1; }
case "$ROCM_TAG" in
  rocm6.[0-4]|rocm7.[02]) T="$ROCM_TAG" ;;
  rocm6.*) T="rocm6.4" ;;
  *) T="rocm7.1" ;;
esac
pip install bitsandbytes
PYTORCH_INDEX_URL="https://download.pytorch.org/whl/${T}"
uv pip install --system -U --force-reinstall \
    torch torchvision torchaudio triton-rocm \
    --index-url "$PYTORCH_INDEX_URL"
uv pip install --system cut-cross-entropy torchao --no-deps
uv pip install --system -U --no-deps "unsloth[amd]" "unsloth_zoo[amd]"
uv pip install --system --no-deps -r "$(python -c 'import pathlib,site;print(next(p for r in [*site.getsitepackages(),site.getusersitepackages()] if (p:=pathlib.Path(r,"studio/backend/requirements/no-torch-runtime.txt")).exists()))')" torchao
uv pip install --system --no-deps -U "tokenizers>=0.22.0,<=0.23.0"


In [ ]:
!uv pip install --system -qqq "transformers==4.56.2"
!uv pip install --system -qqq --upgrade --no-deps "trl==0.22.2"


# 使用 Unsloth 加载模型

使用 LoRA 配置加载 GPT-OSS 20B：

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # 最大上下文长度
lora_rank = 16         # LoRA 等级（较高=更聪明但更慢；4 对于推理任务来说太低）

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-BF16",
    load_in_4bit = False,
    max_seq_length = max_seq_length,
    torch_dtype = torch.bfloat16,
)

# 明确地将模型强制到 cuda
print(f"Model device: {model.device}")
print("模型加载成功！")

# 添加 LoRA 适配器

添加 LoRA 层以进行高效微调：

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 扫雷游戏实现

自定义扫雷环境支持：
- 可定制的棋盘尺寸和地雷数量
- 操作：显示或标记单元格
- 获胜：揭示所有安全单元
- 失败：露出地雷

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Set
import random

@dataclass
class MinesweeperGame:
    rows: int
    cols: int
    num_mines: int
    seed: Optional[int] = None
    _rng: random.Random = field(init = False, repr = False)
    _board: List[List[int]] = field(init = False, repr = False)  # -1 = 我的，0-8 = 计数
    _revealed: Set[Tuple[int, int]] = field(init = False, repr = False, default_factory = set)
    _flagged: Set[Tuple[int, int]] = field(init = False, repr = False, default_factory = set)
    _state: str = field(default = "ongoing", init = False, repr = False)

    def __post_init__(self):
        if self.num_mines >= self.rows * self.cols:
            raise ValueError("Too many mines for board size")
        self._rng = random.Random(self.seed)
        self._board = [[0 for _ in range(self.cols)] for _ in range(self.rows)]
        self._place_mines()
        self._calculate_numbers()

    def _place_mines(self):
        """Place mines randomly on the board"""
        positions = [(r, c) for r in range(self.rows) for c in range(self.cols)]
        mine_positions = self._rng.sample(positions, self.num_mines)
        for r, c in mine_positions:
            self._board[r][c] = -1

    def _calculate_numbers(self):
        """Calculate numbers for each cell based on adjacent mines"""
        for r in range(self.rows):
            for c in range(self.cols):
                if self._board[r][c] == -1:
                    continue
                count = 0
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0:
                            continue
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < self.rows and 0 <= nc < self.cols:
                            if self._board[nr][nc] == -1:
                                count += 1
                self._board[r][c] = count

    def _reveal_cell(self, row: int, col: int) -> bool:
        """Reveal a cell. Returns True if valid move, False if invalid.
        Uses iterative flood-fill to avoid recursion limit on large boards.
        (Issue # 11：是递归的；问题拼写错误：修复了“bself”->“self”）
        """
        if not (0 <= row < self.rows and 0 <= col < self.cols):
            return False
        if (row, col) in self._revealed or (row, col) in self._flagged:
            return False

        stack = [(row, col)]
        while stack:
            r, c = stack.pop()
            if (r, c) in self._revealed:
                continue

            self._revealed.add((r, c))

            # 击中地雷！
            if self._board[r][c] == -1:
                self._state = "failed"
                return True

            # 如果单元格为 0，则自动显示邻居
            if self._board[r][c] == 0:
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0:
                            continue
                        nr, nc = r + dr, c + dc
                        if (0 <= nr < self.rows and 0 <= nc < self.cols
                                and (nr, nc) not in self._revealed
                                and (nr, nc) not in self._flagged):
                            stack.append((nr, nc))

        return True

    def _flag_cell(self, row: int, col: int) -> bool:
        """Flag/unflag a cell. Returns True if valid, False if invalid"""
        if not (0 <= row < self.rows and 0 <= col < self.cols):
            return False
        if (row, col) in self._revealed:
            return False
        
        if (row, col) in self._flagged:
            self._flagged.remove((row, col))
        else:
            self._flagged.add((row, col))
        return True

    def do_action(self, action: dict) -> str:
        """Execute an action and return a status string.

        Returns one of:
          'ok'               - valid move executed
          'mine'             - revealed a mine (game over)
          'win'              - game won after this move
          'invalid_format'   - bad action dict / missing keys / bad types
          'out_of_bounds'    - coordinates outside the board
          'already_revealed' - cell was already revealed
          'flagged_cell'     - tried to reveal a flagged cell
          'invalid_flag'     - tried to flag a revealed cell
          'game_over'        - game was already over before this call

        Only sets state = 'failed' for moves that would change the board state,
        while formatting errors return 'invalid_format' without changing state.
        """
        if self._state != "ongoing":
            return "game_over"

        if not isinstance(action, dict):
            self._state = "failed"
            return "invalid_format"

        action_type = action.get("type")
        row = action.get("row")
        col = action.get("col")

        if action_type not in ["reveal", "flag"] or row is None or col is None:
            self._state = "failed"
            return "invalid_format"

        try:
            row, col = int(row), int(col)
        except (ValueError, TypeError):
            self._state = "failed"
            return "invalid_format"

        if not (0 <= row < self.rows and 0 <= col < self.cols):
            self._state = "failed"
            return "out_of_bounds"

        if action_type == "reveal":
            if (row, col) in self._revealed:
                self._state = "failed"
                return "already_revealed"
            if (row, col) in self._flagged:
                self._state = "failed"
                return "flagged_cell"
            valid = self._reveal_cell(row, col)
        else:
            if (row, col) in self._revealed:
                self._state = "failed"
                return "invalid_flag"
            valid = self._flag_cell(row, col)

        if not valid:
            self._state = "failed"
            return "invalid_format"

        self._check_win()

        if self._state == "failed":
            return "mine"
        if self._state == "success":
            return "win"
        return "ok"

    def _check_win(self):
        """Check if player has won"""
        total_cells = self.rows * self.cols
        safe_cells = total_cells - self.num_mines
        if len(self._revealed) == safe_cells:
            self._state = "success"

    def get_visible_board(self) -> List[List[str]]:
        """Get board state as player sees it"""
        visible = []
        for r in range(self.rows):
            row = []
            for c in range(self.cols):
                if (r, c) in self._flagged:
                    row.append('F')
                elif (r, c) in self._revealed:
                    val = self._board[r][c]
                    row.append('*' if val == -1 else str(val))
                else:
                    row.append('.')
            visible.append(row)
        return visible

    def state(self) -> str:
        return self._state

    def pretty_print(self) -> str:
        """Pretty print the board"""
        visible = self.get_visible_board()
        lines = []
        
        # 标头
        header = "   " + " ".join(f"{i:2d}" for i in range(self.cols))
        lines.append(header)
        lines.append("  " + "─" * (self.cols * 3 + 1))
        
        # 木板
        for r, row in enumerate(visible):
            line = f"{r:2d}│ " + "  ".join(row)
            lines.append(line)
        
        return "\n".join(lines)

# 测试游戏

In [ ]:
# 创建测试游戏
game = MinesweeperGame(rows = 6, cols = 6, num_mines = 5, seed = 42)
print(game.pretty_print())
print(f"State: {game.state()}")

# 测试动作
game.do_action({"type": "reveal", "row": 0, "col": 0})
print("\n显示 (0,0) 后：")
print(game.pretty_print())
print(f"State: {game.state()}")

# JSON 输入/输出格式

## 输入格式（游戏状态）
```json
{
  "board": [
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."],
    [".", ".", ".", ".", ".", "."]
  ],
  "rows": 6,
  "cols": 6,
  "mines": 5
}
```

## 输出格式（动作）
```json
{"type": "reveal", "row": 2, "col": 3}
```
或
```json
{"type": "flag", "row": 1, "col": 4}
```

In [ ]:
import json

def format_state_for_llm(game: MinesweeperGame) -> str:
    """Convert game state to JSON prompt for LLM"""
    state = {
        "board": game.get_visible_board(),
        "rows": game.rows,
        "cols": game.cols,
        "mines": game.num_mines,
        "flags_placed": len(game._flagged),
        "cells_revealed": len(game._revealed),
    }
    
    prompt = f"""You are playing Minesweeper. Analyze the game state and output your next move.

Game state:
{json.dumps(state, indent = 2)}

Legend:
- "." = unrevealed cell
- "F" = flagged cell (suspected mine)
- "0"-"8" = number of adjacent mines
- "*" = revealed mine (game over)

Output your next action as JSON:
{{"type": "reveal", "row": <row_index>, "col": <col_index>}}
or
{{"type": "flag", "row": <row_index>, "col": <col_index>}}

Your action:"""
    
    return prompt

def parse_llm_action(response: str) -> dict:
    """Extract JSON action from LLM response.
    
    Finds all JSON-like objects and returns the LAST one matching the
    expected schema.  LLMs typically reason through options and place
    their final answer at the end, so taking the last valid match is
    more robust than taking the first.
    """
    import re
    best = None
    for match in re.finditer(r'\{[^{}]*\}', response):
        try:
            action = json.loads(match.group())
            if ("type" in action and "row" in action and "col" in action
                    and action["type"] in ["reveal", "flag"]):
                best = action
        except json.JSONDecodeError:
            continue
    return best

# 测试格式化
game = MinesweeperGame(rows = 6, cols = 6, num_mines = 5, seed = 42)
prompt = format_state_for_llm(game)
print(prompt[:500] + "...")

# 训练前测试模型

查看基本模型在不进行微调的情况下的表现：

In [ ]:
from transformers import TextStreamer

game = MinesweeperGame(rows = 6, cols = 6, num_mines = 5, seed = 42)
prompt = format_state_for_llm(game)

# Reasoning_effort = "low" — GRPOTrainer 未通过
# 在训练期间，因此仅在评估时使用它会导致训练/评估不匹配。
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
)

print("=== 基本模型响应 ===")
output = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 128,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

# GRPO奖励功能

定义奖励函数来指导模型的学习：

In [ ]:
import numpy as np


def _is_logically_safe(game, row, col):
    """Check if revealing (row, col) is provably safe via constraint propagation.

    A cell is logically safe if there is at least one adjacent *revealed* number
    cell whose mine count equals its number of flagged neighbors.  That means
    every remaining unrevealed neighbor of that number cell (including (row, col))
    must be safe.
    """
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = row + dr, col + dc
            if not (0 <= nr < game.rows and 0 <= nc < game.cols):
                continue
            if (nr, nc) not in game._revealed:
                continue
            number = game._board[nr][nc]
            if number <= 0:
                continue
            # 计算编号单元格的标记邻居数
            flagged_neighbors = 0
            for dr2 in [-1, 0, 1]:
                for dc2 in [-1, 0, 1]:
                    if dr2 == 0 and dc2 == 0:
                        continue
                    nnr, nnc = nr + dr2, nc + dc2
                    if (0 <= nnr < game.rows and 0 <= nnc < game.cols
                            and (nnr, nnc) in game._flagged):
                        flagged_neighbors += 1
            if flagged_neighbors == number:
                return True  # 所有地雷已被证明是安全的→（行，列）
    return False


def valid_json_reward(completions, **kwargs):
    """Reward valid JSON action format"""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        action = parse_llm_action(response)

        if action is None:
            scores.append(-5.0)  # 格式无效
        else:
            scores.append(1.0)   # 有效格式

    return scores


def gameplay_reward(completions, **kwargs):
    """
    Clean reward function based on defined criteria.

    Reward Criteria:
    1.  Flag cell that IS a mine       → +15
    2.  Flag cell that is NOT a mine    → -10
    3.  Reveal cell that IS a mine      → -25 (game over)
    4.  Reveal cell that is safe        → +10  (+15 if logically deducible)
    5.  Flag already flagged cell       → -8
    6.  Reveal already revealed cell    → -12
    7.  Out of bounds                   → -15
    8.  Total flags > total mines       → -10
    9.  Invalid JSON                    → -10
    10. Win the game                    → +100 (big bonus)
    11. Reveal a flagged cell           → -8  (Issue # 3）
    """
    scores = []

    # 获取从数据集中传递的游戏状态信息
    seeds = kwargs.get("seed", [])
    move_histories = kwargs.get("move_history", [])

    for idx, completion in enumerate(completions):
        response = completion[0]["content"]
        action = parse_llm_action(response)

        # 标准 9：无效 JSON
        if action is None:
            scores.append(-10.0)
            continue

        # 重建精确的游戏状态
        if idx < len(seeds) and idx < len(move_histories):
            seed = seeds[idx]
            move_history_raw = move_histories[idx]

            # 问题 #1：move_history 在数据集中存储为 JSON 字符串
            # 以避免 HF 数据集模式推断破坏字典列表。
            if isinstance(move_history_raw, str):
                move_history = json.loads(move_history_raw)
            else:
                move_history = move_history_raw  # 向后兼容

            # 根据提示将游戏重建到准确状态
            game = MinesweeperGame(rows = 6, cols = 6, num_mines = 5, seed = seed)
            for prev_action in move_history:
                game.do_action(prev_action)

            board = game.get_visible_board()
            row, col = action["row"], action["col"]
            action_type = action["type"]

            # 标准 7：出界
            if not (0 <= row < game.rows and 0 <= col < game.cols):
                scores.append(-15.0)
                continue

            # 获取真实情况和当前状态
            is_mine = game._board[row][col] == -1  # 事实真相（法学硕士隐藏）
            is_revealed = (row, col) in game._revealed
            is_flagged = (row, col) in game._flagged

            # --- 旗帜行动 ---
            if action_type == "flag":
                # 标准 6：无法标记已显示的单元格
                if is_revealed:
                    scores.append(-12.0)
                    continue

                # 标准 5：标记已标记的单元格
                if is_flagged:
                    scores.append(-8.0)
                    continue

                # 标准 8：检查是否超过地雷数量
                current_flag_count = len(game._flagged)
                if current_flag_count >= game.num_mines:
                    scores.append(-10.0)
                    continue

                # 标准 1 和 2：检查标记的单元格是否确实是地雷
                if is_mine:
                    scores.append(15.0)  # 正确的旗帜！
                else:
                    scores.append(-10.0)  # 错误的旗帜
                continue

            # --- 揭示行动 ---
            elif action_type == "reveal":
                # 标准 6：显示已显示的单元格
                if is_revealed:
                    scores.append(-12.0)
                    continue

                # 问题＃3：显示一个标记的单元格 - 正在下降到
                # game.do_action 返回 False → state = "failed" → -25 地雷惩罚。
                if is_flagged:
                    scores.append(-8.0)
                    continue

                # 应用显示操作
                game.do_action(action)

                # 标准 10：赢得比赛
                if game.state() == "success":
                    scores.append(100.0)  # 大奖金！
                    continue

                # 标准 3：发现地雷（游戏结束）
                if game.state() == "failed":
                    scores.append(-25.0)
                    continue

                # 标准 4：揭示安全牢房
                # 问题＃9：逻辑上可推演的动作的奖励
                if _is_logically_safe(game, row, col):
                    scores.append(15.0)  # 可证明安全 → 更高的奖励
                else:
                    scores.append(10.0)  # 随机/不确定的安全揭示
                continue

            # 未知的动作类型
            else:
                scores.append(-10.0)
        else:
            # 倒退
            scores.append(0.0)

    return scores

print("奖励功能创建！")
print("")
print("奖励：")
print("+100：赢得游戏（显示所有安全单元格）")
print("+15：标记实际地雷/逻辑上可推论的安全揭示")
print("+10：显示安全单元（不确定/随机）")
print("+1：有效的 JSON 格式")
print("")
print("处罚：")
print("-25：露出我的（游戏结束）")
print("-15：出界")
print("-12：显示已经显示的单元格")
print("-10：标记非我的、超过我的数量、无效的 JSON")
print("-8：标记已标记/显示标记的单元格")
print("-5：JSON 格式无效")

# 创建训练数据集

生成不同的游戏状态进行训练：

In [ ]:
from datasets import Dataset

def generate_game_states(num_samples = 1000, rows = 6, cols = 6, num_mines = 5,
                         rng_seed = 42):
    """
    Generate EXACTLY num_samples diverse Minesweeper game states.
    
    Mix of:
    - Fresh games (20-30%)
    - Mid-game states (70-80%)
    
    IMPORTANTLY: Stores seed + move_history (as JSON string) so reward
    function can reconstruct the EXACT game state!
    
    Keeps generating until we have exactly num_samples valid ongoing games.
    
    Args:
        rng_seed: Seed for numpy/random RNG for reproducibility (Issue # 10）。
    """
    # 种子 RNG 可实现运行间的再现性
    np.random.seed(rng_seed)
    random.seed(rng_seed)

    dataset_items = []
    attempts = 0
    max_attempts = num_samples * 3  # 安全极限
    
    while len(dataset_items) < num_samples and attempts < max_attempts:
        attempts += 1
        seed = np.random.randint(100000)
        game = MinesweeperGame(rows = rows, cols = cols, num_mines = num_mines, seed = seed)
        
        # 进行 0-5 次随机移动（0 = 新鲜游戏，1-5 = 游戏中期）
        num_moves = np.random.randint(0, 6)
        move_history = []
        
        for _ in range(num_moves):
            board = game.get_visible_board()
            unrevealed = []
            for r in range(rows):
                for c in range(cols):
                    if board[r][c] == '.':
                        unrevealed.append((r, c))
            
            if unrevealed and game.state() == "ongoing":
                r, c = random.choice(unrevealed)
                action = {"type": "reveal", "row": r, "col": c}
                game.do_action(action)
                move_history.append(action)
            else:
                break
        
        # 仅添加正在进行的游戏（跳过失败/已完成的游戏）
        if game.state() == "ongoing":
            prompt_text = format_state_for_llm(game)
            dataset_items.append({
                "prompt": [{"role": "user", "content": prompt_text}],
                "seed": seed,  # 存储种子以重建游戏
                # 序列化为 JSON 字符串以避免 HF 数据集
                # 模式推理将字典列表转换为列表字典
                "move_history": json.dumps(move_history),
            })
    
    return Dataset.from_list(dataset_items)

# 生成训练数据集
print("正在生成训练数据集...")
dataset = generate_game_states(num_samples = 1000, rows = 6, cols = 6, num_mines = 5)
print(f"Created EXACTLY {len(dataset)} training examples (all ongoing games)")

# 计算新鲜与游戏中期
fresh_count = sum(1 for item in dataset if item["move_history"] == "[]")
print(f"  Fresh games: {fresh_count} ({fresh_count/len(dataset)*100:.1f}%)")
print(f"  Mid-game states: {len(dataset) - fresh_count} ({(len(dataset)-fresh_count)/len(dataset)*100:.1f}%)")

# 显示示例
print("\n训练提示示例：")
print(dataset[0]["prompt"][0]["content"][:400] + "...")
print(f"Seed: {dataset[0]['seed']}, Previous moves: {len(json.loads(dataset[0]['move_history']))}")

# 配置 GRPO 培训

使用所有超参数设置 GRPO 训练器：

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# 计算最大长度
max_prompt_length = 600   # JSON状态提示
max_completion_length = max_seq_length - max_prompt_length

# GRPO 配置
training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,  # 4 每次更新提供 16 个有效完成
    num_generations = 4,  # 每个状态生成 4 个操作
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 500,      # 根据计算预算进行调整
    save_steps = 100,
    report_to = "none",
    output_dir = "minesweeper_custom_outputs",
)

print("训练配置：")
print(f"  Max steps: {training_args.max_steps}")
print(f"  Generations per state: {training_args.num_generations}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  LoRA rank: {lora_rank}")

In [ ]:
from transformers import TrainerCallback

class MinesweeperEvalCallback(TrainerCallback):
    """Periodically play games during training and log win rate.
    (Issue # 8：原始笔记本中没有验证/奖励跟踪。）
    """

    def __init__(self, eval_every_steps = 50, num_games = 5):
        self.eval_every_steps = eval_every_steps
        self.num_games = num_games

    def on_step_end(self, args, state, control, model = None, processing_class = None, **kwargs):
        if state.global_step % self.eval_every_steps != 0:
            return

        tokenizer = processing_class
        if tokenizer is None or model is None:
            return

        # 暂时将模型设置为 eval 模式
        was_training = model.training
        model.eval()

        wins = 0
        for i in range(self.num_games):
            game = MinesweeperGame(rows = 6, cols = 6, num_mines = 5, seed = 10000 + i)
            moves = 0
            while game.state() == "ongoing" and moves < 50:
                prompt = format_state_for_llm(game)
                text = tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    tokenize = False,
                    add_generation_prompt = True,
                )
                output = model.generate(
                    **tokenizer(text, return_tensors = "pt").to(model.device),
                    temperature = 0.7,
                    max_new_tokens = 128,
                    do_sample = True,
                )
                response = tokenizer.decode(output[0], skip_special_tokens = True)
                action = parse_llm_action(response)
                if action is None:
                    break
                game.do_action(action)
                moves += 1
            if game.state() == "success":
                wins += 1

        win_rate = wins / self.num_games
        print(f"\n[Eval @ step {state.global_step}] Win rate: {wins}/{self.num_games} ({win_rate*100:.0f}%)\n")

        if was_training:
            model.train()

eval_callback = MinesweeperEvalCallback(eval_every_steps = 50, num_games = 5)
print("创建评估回调：每 50 步玩 5 场游戏")

# 训练模型

使用奖励函数开始 GRPO 训练：

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        valid_json_reward,   # 奖励有效的JSON格式
        gameplay_reward,     # 奖励良好的游戏玩法
    ],
    args = training_args,
    train_dataset = dataset,
    callbacks = [eval_callback],  # 问题#8：定期游戏评估
)

print("开始训练...")
trainer.train()

# 测试训练模型

评估微调后的模型：

In [ ]:
# 新游戏测试
test_game = MinesweeperGame(rows = 6, cols = 6, num_mines = 5, seed = 999)
test_prompt = format_state_for_llm(test_game)

# 问题#7：删除 Reasoning_effort = “low” 以保证训练/评估一致性
test_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize = False,
    add_generation_prompt = True,
)

print("=== 训练模型响应 ===")
output = model.generate(
    **tokenizer(test_text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 128,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

# 解析和测试动作
response_text = tokenizer.decode(output[0])
action = parse_llm_action(response_text)
print(f"\nParsed action: {action}")

if action:
    test_game.do_action(action)
    print(f"\nGame state after action: {test_game.state()}")
    print(test_game.pretty_print())

# 评估：玩完整游戏

在多个完整游戏上测试模型：

In [ ]:
def play_full_game(model, tokenizer, rows = 6, cols = 6, num_mines = 5, seed = None, max_moves = 50):
    """Play a complete Minesweeper game with the model"""
    game = MinesweeperGame(rows = rows, cols = cols, num_mines = num_mines, seed = seed)
    moves = 0
    
    while game.state() == "ongoing" and moves < max_moves:
        # 获取当前状态
        prompt = format_state_for_llm(game)
        # 删除了 Reasoning_effort = “low” 以保证训练/评估的一致性
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize = False,
            add_generation_prompt = True,
        )
        
        # 生成动作
        output = model.generate(
            **tokenizer(text, return_tensors = "pt").to("cuda"),
            temperature = 0.7,
            max_new_tokens = 128,
            do_sample = True,
        )
        
        response = tokenizer.decode(output[0])
        action = parse_llm_action(response)
        
        if action is None:
            break  # 无效动作
        
        game.do_action(action)
        moves += 1
    
    return game, moves

# 评估 100 场比赛的具有统计意义的胜率
NUM_EVAL_GAMES = 100
print(f"Evaluating model on {NUM_EVAL_GAMES} games...\n")
wins = 0
total_moves = 0

for i in range(NUM_EVAL_GAMES):
    game, moves = play_full_game(model, tokenizer, seed = i)
    result = game.state()

    if result == "success":
        wins += 1
    # 仅打印前 10 场比赛以及任何胜利的个人结果
    if i < 10 or result == "success":
        tag = "WIN" if result == "success" else "LOSS"
        print(f"Game {i+1}: {tag} ({result}) after {moves} moves")

    total_moves += moves

if NUM_EVAL_GAMES > 10:
    print(f"... (showing first 10 + wins; {NUM_EVAL_GAMES} total)")

print(f"\nResults:")
print(f"  Win rate: {wins}/{NUM_EVAL_GAMES} ({wins/NUM_EVAL_GAMES*100:.1f}%)")
print(f"  Average moves: {total_moves/NUM_EVAL_GAMES:.1f}")

# 保存模型

保存训练好的模型以供竞赛提交：

In [ ]:
# 保存 LoRA 适配器
model.save_pretrained("gpt_oss_lora")
tokenizer.save_pretrained("gpt_oss_lora")

print("模型保存到：my_minesweeper_model/")

# 可选：以 16 位保存合并模型
if False:
    model.save_pretrained_merged(
        "gpt_oss_finetune_16bit",
        tokenizer,
        save_method = "merged_16bit"
    )

# 可选：推至拥抱面部中心
if False:
    model.push_to_hub_merged(
        "your-username/minesweeper-gpt-oss",
        tokenizer,
        save_method = "lora",
        token = "YOUR_HF_TOKEN"
    )

# 温馨提示

## 改进你的模型：

1. **调整奖励功能**
   - 增加逻辑推理奖励
   - 添加随机移动的惩罚
   - 奖励标记正确的地雷

2. **调整超参数**
   - 增加“max_steps”以进行更长时间的训练
   - 调整`learning_rate`（尝试1e-5到1e-4）
   - 增加“lora_rank”以获得更多容量
   - 调整“num_ Generations”（2-8）

3. **更好的训练数据**
   - 生成更多样的状态
   - 包括更困难的场景（更多地雷）
   - 添加需要逻辑推演的状态

4. **先进技术**
   - 奖励功能的多步推出
   - 课程学习（简单→硬板）
   - 集成多个模型

## 有用的策略：
- 尝试不同的奖励函数
- 在训练期间尝试不同的板尺寸
- 分析失败的游戏以提高奖励
- 评估期间使用温度采样
我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  此笔记本和所有 Unsloth 笔记本均已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>